In [10]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. 환경(패키지 install)
- pip install langchain-ollama
- ollama pull exaone3.5:2.4b
- ollama pull llama3.2:1b

# 2. LLM 답변 생성

## 2.1 OpenAI SDK를 사용

In [16]:
%%time
from openai import OpenAI
# ollama의 docs에서 base_url

client = OpenAI(
    api_key="amuguna", # 필수 인자라 생략할 수 없어서 아무거나 입력
    base_url="http://localhost:11434/v1"
)
response = client.chat.completions.create(
    model = "exaone3.5:2.4b", # ollama list에 나오는 모델 이름 그대로
    messages=[
        {'role':'system', 'content':'너는 친절하게 대답해 주는 비서야'},
        {'role':'user', 'content':'2020년 월드 시리즈 누가 우승했어?'}
    ],
    temperature=0.5, # 0~0.3:일관되고 예측가능한 답변 / 0.8~1 : 창의적이고 다양한 답변
    stream=False
)

CPU times: total: 31.2 ms
Wall time: 7.17 s


In [12]:
response.choices[0].message.content

'2020년 월드 시리즈에서는 **로스앤젤레스 다저스**가 우승했습니다. \n\n다저스는 탬파베이 레이스를 4승 2패로 꺾고 시리즈 정상에 올랐으며, 이는 다저스가 1988년 이후 32년 만에 거둔 월드 시리즈 우승이었습니다. (당시 코로나19 팬데믹으로 인해 시즌은 무관중과 특수 방역 프로토콜 하에 진행되었습니다.)'

In [13]:
response.usage # 사용 토큰수

CompletionUsage(completion_tokens=81, prompt_tokens=70, total_tokens=151, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

## 2.2 LangChain 사용

In [18]:
%%time
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
llm = ChatOllama(
    model="exaone3.5:2.4b"
)
messages = [
    SystemMessage(content=
                  '너는 전문 지식과 실무 감각을 갖춘 비서다. 질문에 맞는 전문가 역할을 맡아 핵심부터 짧고 정확하게 답하라. 근거와 실행 방법을 제시하고, 불필요한 설명·아첨은 빼며, 모르는 것은 추측하지 마라.'),
    HumanMessage(content='2020년 월드 시리즈 누가 우승했어?')
]
ai_message = llm.invoke(messages)
print(ai_message.content)

2020년 월드 시리즈 우승 팀은 **뉴욕 양키스**입니다. 

* **실행 방법**: 이 정보는 스포츠 역사 기록을 확인하거나 공식 MLB 웹사이트에서 확인 가능합니다.
CPU times: total: 62.5 ms
Wall time: 2.95 s
